# Modelagem da Construção Civil com Redes de Petri Coloridas (RPC / CPN - Colored Petri Nets)



**Sumário**

- [1. Introdução](#sec-1)
- [2. Das redes ordinárias às redes coloridas](#sec-2)
- [3. Tokens e representação do estado](#sec-3)
- [4. Declarações, atividades e equipes](#sec-4)
- [5. Tipos compostos](#sec-5)
- [6. Listas e composição de equipes](#sec-6)
- [7. Representação do tempo](#sec-7)
- [8. Variáveis e durações](#sec-8)
- [9. Funções e seleção do maior valor](#sec-9)
- [10. Agregação dos tempos de predecessoras](#sec-10)
- [11. Sincronização e caminho crítico](#sec-11)
- [12. Execução da rede e algoritmo do CPM](#sec-12)
- [13. Execução de fluxos concorrentes](#sec-13)
- [14. Alocação e reutilização de recursos](#sec-14)
- [15. Integração entre CPM e RPC](#sec-15)
- [16. Integrações e aplicações avançadas](#sec-16)
- [17. Conclusão](#sec-17)
- [18. Referências](#sec-18)
- [19. Apêndice: Redes de Petri Coloridas e Linguagens de Programação](#sec-19)


<a id="sec-1"></a>


## 1. Introdução

As **Redes de Petri Coloridas (RPC / Colored Petri Nets — CPN)** constituem uma extensão formal das Redes de Petri clássicas voltada para modelagem de sistemas concorrentes, distribuídos, sincronizados e temporalmente dependentes. Enquanto que nas **Redes de Petri ordinárias os tokens são indistinguíveis**,  as **CPN introduzem tipagem forte, variáveis, expressões, listas, estruturas compostas e temporização**, permitindo representar sistemas reais de elevada complexidade sem gerar explosão estrutural da rede.

Na construção civil, isso é particularmente importante porque **a execução de uma obra envolve simultaneamente precedência lógica, compartilhamento de equipes, sincronização operacional, concorrência entre frentes de trabalho, propagação temporal e restrições de recursos**. 

Assim, **a CPN deixa de representar apenas um fluxo abstrato de atividades e passa a representar dinamicamente o estado operacional da obra**.

O modelo utilizado nesta aula implementa uma **integração entre Redes de Petri Coloridas** e o **Método do Caminho Crítico (CPM — Critical Path Method)** e, além disso, permite representar dependências entre atividades, propagação temporal automática, sincronização entre fluxos paralelos, compartilhamento de equipes, cálculo dinâmico do caminho crítico e execução concorrente de atividades.

A CPN utilizada no exemplo de aplicação utilizado está no arquivo [GPCrev*.cpn](https://raw.githubusercontent.com/UniRobotica/Petri-nets/refs/heads/main/Exemplos/GPCrev9_Aula.cpn) disponível no repositório deste curso.
Este modelo está relacionado ao exemplo de PERT/CPM utilizado em [gerenciamento de projetos](https://gpc.unirobotica.com.br/notebooks/dominios/planejamento/pertcpm-construcao-civil.html).

---



<a id="sec-2"></a>

## 2. Das redes ordinárias às redes coloridas


### 2.1 Fundamentos de RPC: Elementos e dinâmica de uma RPC

**RPC** é a sigla em português para **Rede de Petri Colorida**; **CPN** é a sigla em inglês para o mesmo formalismo. A estrutura de lugares, transições e arcos é mantida, mas as marcas passam a ter valores tipados.

| Elemento | Pergunta que ajuda a responder |
|---|---|
| **Lugar** | Em que condição uma entidade ou recurso se encontra? |
| **Token ou marca** | Qual entidade, informação ou recurso está nessa condição? |
| **Transição** | Que evento pode alterar essa condição? |
| **Arco e sua inscrição** | Quais valores são consumidos ou produzidos? |
| **Guarda** | Que condição adicional deve ser verdadeira? |
| **Marcação** | Quais tokens estão distribuídos em todos os lugares neste momento? |

Para disparar uma transição, o simulador associa valores às variáveis (**binding**), verifica se os tokens exigidos existem nas entradas e testa a guarda. O disparo consome os tokens especificados e produz os valores definidos nas saídas, gerando outra marcação.

**Exemplo simples:** uma transição que recebe uma solicitação e um recurso disponível precisa dos dois tokens. Se faltar o recurso, a solicitação permanece aguardando. Uma guarda verdadeira, sozinha, não basta para habilitar a transição.

Essa dinâmica continua válida nas redes coloridas: os tipos e as expressões acrescentam informação às regras de disparo ([cpntools.org](https://cpntools.org/2018/01/12/inscriptions-and-expressions/)).


### 2.2 Redes de Petri Coloridas (CPN)

As CPN adicionam uma camada semântica importante às Redes de Petri tradicionais. Em vez de tokens (marcas) equivalentes e sem significado interno, os **tokens passam a transportar informação estruturada**. Dessa forma, o estado do sistema deixa de estar exclusivamente representado na topologia da rede e passa a ser encapsulado nos próprios tokens.

Isso reduz significativamente a complexidade estrutural do modelo. Em problemas reais, como **planejamento de obras**, uma Rede de Petri clássica exigiria uma quantidade muito grande de lugares e transições para representar diferentes estados operacionais. Nas CPNs, grande parte dessa lógica é abstraída através de tipos, variáveis e funções.

O modelo [GPCrev*.cpn](https://raw.githubusercontent.com/UniRobotica/Petri-nets/refs/heads/main/Exemplos/GPCrev9_Aula.cpn) não representa apenas o fluxo e precedências entre atividades como no tradicional CPM. Ele representa simultaneamente estado da obra, equipes alocadas, duração das atividades, tempo acumulado, sincronização operacional, propagação temporal e restrições de recursos. Assim, a rede torna-se um simulador discreto de execução da obra.

---



<a id="sec-3"></a>

## 3. Tokens e representação do estado


### 3.1 Fundamentos de RPC: Semântica dos tokens: valor, cor e marcação

Em RPC, a **cor de um token é seu valor**, e não a cor usada no desenho. O **conjunto de cores** (*color set*) define os valores permitidos: inteiros, nomes de estados, listas ou estruturas compostas. Cada lugar possui um tipo que limita os tokens que pode receber.

O significado resulta da combinação entre **valor do token** e **lugar onde ele está**. O valor pode identificar uma solicitação; o lugar pode indicar que ela está aguardando ou concluída. Por isso, o estado global é a marcação completa da rede, mesmo quando os tokens carregam muitos dados.

A marcação de um lugar é um **multiconjunto**: valores iguais podem aparecer várias vezes. Em um lugar de tipo inteiro, por exemplo:

```sml
2`7 ++ 1`9
```

Essa inscrição representa **três tokens**: dois com valor `7` e um com valor `9`. O número antes da crase indica a multiplicidade; `++` combina multiconjuntos. `empty` representa ausência de tokens.

**Leitura de um disparo:** considere um arco de entrada com `x`, uma guarda `[x < 10]` e um arco de saída com `x + 1`, conectados a lugares do tipo inteiro. Se o binding selecionar `x = 7`, a guarda será verdadeira: o disparo consome um token `7` e produz um token `8`. As demais marcas permanecem nos lugares. [cpntools.org/2018/01/09/guards/](https://cpntools.org/2018/01/09/guards/)

Um token pode ter vários campos e continuar sendo **uma única marca**. No exemplo da obra a seguir, leia a tupla inteira como o valor de um token; cada posição descreve um atributo da entidade representada. [cpntools.org/2018/01/09/multisets/](https://cpntools.org/2018/01/09/multisets/)


### 3.2 Estrutura Semântica dos Tokens

No modelo utilizado, o token representa uma entidade operacional completa da obra. O estado do projeto é transportado dentro do próprio token através da estrutura:

```sml
(id, ativ, eqp_list, dur, t)
```

Essa estrutura representa simultaneamente:

| Campo | Significado |
|---|---|
| `id` | identificador da instância do projeto |
| `ativ` | atividade atual |
| `eqp_list` | equipes alocadas |
| `dur` | duração planejada |
| `t` | instante temporal acumulado |

Nessa abordagem, o token deixa de ser apenas uma marca abstrata e passa a representar efetivamente o estado operacional da obra.
Isso transforma a CPN em um modelo temporal, workflow executável, simulador de eventos discretos e representação dinâmica do CPM.

---



<a id="sec-4"></a>

## 4. Declarações, atividades e equipes


### 4.1 Fundamentos de RPC: Declarações, tipos, variáveis, valores e funções

As declarações fornecem o vocabulário usado nas inscrições da rede. Em CPN ML, baseado em Standard ML, quatro construções têm papéis diferentes:

| Construção | Papel | Exemplo didático |
|---|---|---|
| `colset` | Define um tipo de valor permitido | `colset QTD_DID = int;` |
| `var` | Declara uma variável que receberá um valor em um disparo | `var q : QTD_DID;` |
| `val` | Associa um nome a um valor | `val limiteDid = 3;` |
| `fun` | Define um cálculo que retorna um valor | `fun dobroDid(x:int) = 2*x;` |

O tipo deve estar declarado antes de ser usado. A inscrição de um arco precisa produzir valores compatíveis com o tipo do lugar conectado.

**Como ler:** `var q : QTD_DID;` significa que `q` pode representar um valor de `QTD_DID`; não significa que um token já foi criado. Os tokens iniciais são definidos pela **marcação inicial dos lugares**.

Os trechos didáticos desta aula explicam a sintaxe; as declarações da aplicação são apresentadas na sequência. [cpntools.org/2018/01/16/declarations](https://cpntools.org/2018/01/16/declarations/)


### 4.2 Declarações do Modelo

No CPN IDE, as declarações são realizadas na aba `Declarations`. Essas declarações definem os tipos, variáveis, listas, funções e estruturas compostas utilizadas na rede. O modelo [GPCrev*.cpn](https://raw.githubusercontent.com/UniRobotica/Petri-nets/refs/heads/main/Exemplos/GPCrev9_Aula.cpn) utiliza declarações relativamente avançadas para representar a lógica da construção civil.

---



### 4.3 Fundamentos de RPC: Conjuntos enumerados e estados possíveis

Um **conjunto enumerado** define alternativas com nomes explícitos. É útil quando uma entidade só pode assumir estados conhecidos:

```sml
colset FASE_DID = with AGUARDA_DID | EXECUTA_DID | TERMINA_DID;
```

Leia `with` como “os valores possíveis são” e `|` como separador das alternativas. Um token desse tipo pode ter valor `AGUARDA_DID`, por exemplo. Um nome entre aspas seria uma *string*, que é outro tipo.

**A enumeração define quais estados existem; as transições e os arcos definem como mudar entre eles.** A ordem dos nomes na declaração não determina a sequência de execução.

No exemplo da obra, essa ideia será usada para nomear as atividades. Identificar uma atividade no token e representar sua posição na rede são decisões complementares de modelagem. [pntools.org/2018/01/09/enumerated-color-sets](https://cpntools.org/2018/01/09/enumerated-color-sets/)


### 4.4 Conjunto de Atividades

```sml
colset ATIV =
with INICIO | A | B | C | D | E | E1 | F | G | H | I | J | K | L | M | N | FIM;
```

Essa declaração define um conjunto enumerado de estados possíveis da obra. Cada valor representa uma etapa do projeto.

| Código | Atividade |
|---|---|
| `INICIO` | início do projeto |
| `A` | escavação |
| `B` | fundação |
| `C` | paredes |
| `D` | telhado |
| `E` | encanamento exterior |
| `E1` | refinamento/subetapa hidráulica |
| `F` | encanamento interior |
| `G` | muros |
| `H` | pintura exterior |
| `I` | instalação elétrica |
| `J` | divisórias |
| `K` | piso |
| `L` | pintura interior |
| `M` | acabamento exterior |
| `N` | acabamento interior |
| `FIM` | conclusão da obra |

Um aspecto importante é que a atividade não está representada apenas estruturalmente na rede. **O estado da atividade "viaja" dentro do token**. Isso reduz redundâncias estruturais e permite criar modelos muito mais compactos e escaláveis.

---



### 4.5 Fundamentos de RPC: Tipos de recursos e disponibilidade

Uma RPC pode representar recursos como pessoas, máquinas ou documentos. Um tipo enumerado identifica **categorias** de recursos: dois tokens com o mesmo valor podem representar duas unidades intercambiáveis da mesma categoria.

Se for necessário distinguir indivíduos, cada recurso também precisa de um identificador. Saber que existem dois pedreiros é diferente de saber quais profissionais foram alocados.

Há ainda duas informações distintas:

| Informação | O que representa |
|---|---|
| Recurso indicado em um token de atividade | Equipe requerida ou associada à atividade |
| Token em um lugar de recursos disponíveis | Unidade que pode ser reservada por uma atividade |

O controle de capacidade depende das regras da rede: uma atividade deve reservar os recursos necessários e liberá-los ao terminar. **Declarar os nomes das equipes não cria, por si só, esse controle.** Essa distinção orienta a leitura das equipes da aplicação.


### 4.6 Modelagem das Equipes

```sml
colset EQUIPE =
with NSA | ENG | MESTRE | PEDR | ENC | PINTOR | CARP | ELET;
```

Essa declaração representa os recursos humanos da obra.

| Código | Equipe |
|---|---|
| `NSA` | nenhuma equipe associada |
| `ENG` | engenheiro |
| `MESTRE` | mestre de obras |
| `PEDR` | pedreiro |
| `ENC` | encanador |
| `PINTOR` | pintor |
| `CARP` | carpinteiro |
| `ELET` | eletricista |

Essa parte do modelo introduz restrições operacionais. **Em modelos simplificados de precedência de um CPM, normalmente apenas as dependências lógicas são representadas. No entanto, em uma obra real, uma atividade depende não apenas da conclusão da predecessora, mas também da disponibilidade das equipes necessárias.** Assim, a o modelo CPN passa a representar simultaneamente precedência lógica, disponibilidade operacional, compartilhamento de recursos, troca de equipes e sincronização entre frentes de trabalho.

---



<a id="sec-5"></a>

## 5. Tipos compostos


### 5.1 Fundamentos de RPC: Produto cartesiano e tuplas tipadas

O **produto cartesiano** combina tipos em uma estrutura de posições fixas, chamada **tupla**. Cada posição aceita valores do tipo correspondente.

```sml
colset COD_DID = string;
colset QUANT_DID = int;
colset ITEM_DID = product COD_DID * QUANT_DID;
```

Um valor de `ITEM_DID` é `("L1",3)`: código seguido de quantidade. A ordem importa; `(3,"L1")` não pertence a esse produto, pois inverte os tipos.

Nas declarações, `*` combina tipos. Nos valores, parênteses e vírgulas agrupam os campos. Assim, `product` não multiplica quantidades nem cria vários tokens: ele define **o formato de cada token**.

Um arco de entrada pode usar o padrão `(codigo,quantidade)` para associar variáveis aos campos de uma tupla. O arco de saída pode preservar alguns valores e calcular outros. Na aplicação, o mesmo princípio reúne os cinco atributos do projeto. [F5](https://cpntools.org/2018/01/09/product-color-sets/)


### 5.2 Estruturas Compostas e Produto Cartesiano

O núcleo semântico do modelo é definido por:

```sml
colset PROJ = product ID * ATIV * EQP_LIST * DUR * TEMPO;
```

Essa declaração define o tipo principal dos tokens do sistema.

Formalmente:

```text
PROJ = ID × ATIV × EQP_LIST × DUR × TEMPO
```

Cada token passa a representar simultaneamente:

- identidade do projeto;
- estado atual da execução;
- equipes alocadas;
- duração da atividade;
- instante temporal acumulado.

Essa estrutura é fundamental para compreender por que o modelo implementa uma versão dinâmica do CPM.
O token deixa de representar apenas “uma marca na rede” e passa a representar uma entidade operacional completa da obra.

---



<a id="sec-6"></a>

## 6. Listas e composição de equipes


### 6.1 Fundamentos de RPC: Listas, repetição e quantidade de tokens

Uma **lista** reúne uma quantidade variável de elementos de um mesmo tipo. Ao contrário da tupla, sua quantidade de posições pode variar; a ordem dos elementos faz parte do valor.

```sml
colset RECURSO_DID = with OPERADOR_DID | TECNICO_DID;
colset LISTA_DID = list RECURSO_DID;
```

São valores de `LISTA_DID`: `[]`, `[OPERADOR_DID]` e `[OPERADOR_DID,OPERADOR_DID,TECNICO_DID]`. A repetição conserva a informação de quantidade.

| Representação | Interpretação |
|---|---|
| ``1`[OPERADOR_DID,OPERADOR_DID]`` | Um token cujo valor é uma lista com dois elementos |
| ``2`OPERADOR_DID`` | Dois tokens com o mesmo valor, em um lugar do tipo `RECURSO_DID` |

Ter uma lista dentro de um token não equivale a ter recursos separados circulando na rede. A seguir, a lista descreve a composição da equipe associada à atividade. [F6](https://cpntools.org/2018/01/09/list-color-sets/)


### 6.2 Listas de Equipes

O modelo também utiliza listas de equipes, por exemplo:

```sml
[ENG,MESTRE,PEDR,PEDR,PEDR]
```

Essa estrutura representa explicitamente a composição operacional necessária para determinada atividade.

Por exemplo:

```sml
(id,A,[ENG,MESTRE,PEDR,PEDR,PEDR],2,t)
```

representa:

- atividade de escavação;
- participação de engenheiro;
- mestre de obras;
- três pedreiros;
- duração igual a 2 unidades de tempo;
- início no instante `t`.

Esse nível de detalhamento aproxima o modelo da realidade operacional da construção civil, permitindo analisar restrições de recursos, compartilhamento de equipes e impactos operacionais de atrasos.

---



<a id="sec-7"></a>

## 7. Representação do tempo


### 7.1 Fundamentos de RPC: Tempo como dado e tempo de simulação

Uma RPC pode tratar informações temporais de duas maneiras:

| Representação | Efeito |
|---|---|
| Campo numérico no valor do token | Permite calcular e transportar instantes ou durações |
| Carimbo temporal (*timestamp*) de um token em tipo `timed` | Determina quando o token fica disponível ao simulador |

Em um exemplo temporal independente:

```sml
colset TAREFA_T_DID = string timed;
(* Exemplo de inscrição em arco de saída: *)
1`"tarefa" @+ 4
```

Se o relógio da simulação estiver em `10`, o token produzido ficará disponível em `14`, considerando apenas esse atraso. O disparo é um evento; a espera é representada pela disponibilidade futura dos tokens.

Já calcular `t + dur` dentro de uma tupla atualiza **um dado**, sem impor automaticamente uma espera ao simulador. No arquivo da aplicação, `PROJ` não é declarado `timed`: a propagação apresentada deve ser lida como cálculo dos instantes nos campos dos tokens.

Também é preciso definir a unidade usada. Uma duração igual a `4` pode representar dias, horas ou outra unidade, conforme a interpretação do modelo. [F7](https://cpntools.org/2018/01/09/timed-color-sets/) · [F8](https://cpntools.org/2018/01/09/timed-multisets/)


### 7.2 Estrutura Temporal e CPM

O modelo implementa explicitamente conceitos do Método do Caminho Crítico (CPM). O CPM tradicional calcula:

- Early Start (ES);
- Early Finish (EF);
- Late Start (LS);
- Late Finish (LF);
- folgas;
- caminho crítico.

No **Método do Caminho Crítimo (CPM) clássico, através do avanço direto da rede calcula-se os tempos mais cedo**. **O avanço reverso calcula os tempos mais tarde.** O caminho crítico corresponde às atividades cuja folga é zero.
No **modelo CPN, essa propagação temporal é realizada dinamicamente** através dos próprios tokens e funções temporais.

---



<a id="sec-8"></a>

## 8. Variáveis e durações


### 8.1 Fundamentos de RPC: Variáveis e associação de valores no disparo

Uma variável de RPC funciona como um nome que será associado a um valor durante a avaliação de uma transição. Essa associação é o **binding**; não é uma atribuição global compartilhada por toda a rede.

Considere um token com valor `("L1",4)` e um arco de entrada com padrão `(codigo,duracao)`. Ao selecionar esse token, o simulador obtém `codigo = "L1"` e `duracao = 4`. O arco de saída pode reutilizar esses valores em outra expressão.

Se a mesma variável aparecer em dois arcos de entrada **da mesma transição**, seu valor deve ser compatível com ambos. Isso permite, por exemplo, combinar informações que pertencem à mesma instância de projeto.

**Declarar não é inicializar:** `var duracao : DUR;` informa o tipo, mas não fixa uma duração. O valor precisa ser obtido dos tokens ou de outra associação válida. Para comparar cenários, é necessário alterar os dados ou parâmetros que efetivamente alimentam essas expressões e executar novamente o modelo. [F9](https://cpntools.org/2018/01/09/variables/)


### 8.2 Declaração das Durações

```sml
var durA,durB,durC,durD,durE,durE1,durF,durG,
durH,durI,durJ,durK,durL,durM,durN: DUR;
```

As durações foram parametrizadas para permitir:

- simulação de cenários;
- análise de atrasos;
- alteração de produtividade;
- análise de sensibilidade;
- recalculação automática do caminho crítico.

Isso é extremamente importante porque o modelo não fica rigidamente preso a uma única configuração temporal.

---



<a id="sec-9"></a>

## 9. Funções e seleção do maior valor


### 9.1 Fundamentos de RPC: Funções e expressões condicionais

Uma função reúne um cálculo que pode ser reutilizado nas inscrições da rede. Em SML, sua forma básica é:

```sml
fun somaDid(a:int, b:int) = a + b;
```

`fun` inicia a definição; `somaDid` é o nome; `(a:int,b:int)` descreve os argumentos; a expressão após `=` fornece o resultado. `somaDid(3,5)` retorna `8`.

Uma expressão `if condicao then valor1 else valor2` também retorna um valor. Os dois resultados alternativos precisam ter o mesmo tipo.

Uma função de máximo compara os valores recebidos. Ela só fornece o maior **término** quando seus argumentos representam términos; comparar apenas durações não considera possíveis diferenças nos instantes de início.

**A função calcula; a estrutura da rede determina quando esse cálculo participa de um disparo.** Esse é o papel a observar na função apresentada a seguir. [F10](https://cpntools.org/2018/01/09/functions-declarations-and-control-structures/)


### 9.2 Função `maxDur`

```sml
fun maxDur(a:int, b:int) =
if a > b then a else b;
```

Essa função implementa o conceito central do CPM relacionado à propagação dos tempos mais cedo.

Quando uma atividade possui múltiplas predecessoras, seu início mais cedo depende do maior término entre todas as predecessoras.

Formalmente:

```text
ES(j) = max(EF predecessoras)
```

Como:

```text
EF = ES + duração
```

a função `maxDur` passa a representar a lógica de seleção do maior caminho temporal.

---



<a id="sec-10"></a>

## 10. Agregação dos tempos de predecessoras


### 10.1 Fundamentos de RPC: Padrões de listas, pares e agregação de tempos

Funções em SML podem receber estruturas e separar seus componentes por **casamento de padrões** (*pattern matching*). No padrão `[(t1,d1),(t2,d2)]`:

- os colchetes indicam uma lista;
- cada par entre parênteses contém início e duração;
- o padrão exige exatamente dois pares.

Para os valores `[(3,4),(5,6)]`, as associações são:

| Par | Início | Duração | Término calculado |
|---|---:|---:|---:|
| Primeiro | 3 | 4 | 7 |
| Segundo | 5 | 6 | 11 |

Escolher o maior término resulta em `11`, instante em que as duas atividades já podem estar concluídas, considerando essas durações e esses inícios.

Calcular esse número e exigir dois tokens de entrada são operações complementares: **a função agrega tempos; a transição sincroniza os fluxos**. Para atender quantidades diferentes de predecessoras, o padrão ou a função precisa ser ampliado. [www.cs.cornell.edu/courses/cs312/2004fa/lectures/lecture3.htm](https://www.cs.cornell.edu/courses/cs312/2004fa/lectures/lecture3.htm)


### 10.2 Função `max_sum_time`

```sml
fun max_sum_time [(t1, d1), (t2, d2)] =
if t1 + d1 >= t2 + d2 then t1 + d1 else t2 + d2;
```

Essa função implementa explicitamente o cálculo do término mais cedo entre dois fluxos concorrentes.

Ela calcula:

```text
max(t1+d1, t2+d2)
```

onde:

| Variável | Significado |
|---|---|
| `t1` | início da atividade 1 |
| `d1` | duração da atividade 1 |
| `t2` | início da atividade 2 |
| `d2` | duração da atividade 2 |

O resultado representa o instante no qual todas as predecessoras já concluíram suas execuções.

Isso implementa diretamente a lógica de sincronização temporal do CPM.

---



<a id="sec-11"></a>

## 11. Sincronização e caminho crítico


### 11.1 Fundamentos de RPC: Sincronização de predecessoras e identidade dos tokens

Uma transição com dois lugares de entrada pode representar uma **junção AND**: precisa receber a evidência das duas condições para disparar. A presença de um token em apenas um dos lugares é insuficiente.

Ao combinar tokens, é necessário verificar também sua identidade. Se duas obras usam a mesma rede, a conclusão de uma etapa da obra `P1` deve ser combinada com a etapa correspondente da própria `P1`. Uma variável de identificação compartilhada nos arcos de entrada pode impor essa correspondência.

Depois de reunir as predecessoras, o início permitido é o maior dos términos, desde que não existam outras restrições.

**Predecessora que controla uma junção** e **atividade do caminho crítico completo** são conceitos relacionados, mas distintos. O maior término determina a espera naquele encontro; identificar a criticidade em todo o projeto exige considerar os demais caminhos e as folgas. O exemplo seguinte mostra a lógica local dessa sincronização.


### 11.2 Relação com o Caminho Crítico

Considere o trecho do modelo:

```text
F -> J
I -> J
```

A atividade `J` depende simultaneamente de:

- encanamento interno (`F`);
- instalação elétrica (`I`).

Supondo:

| Atividade | ES | Duração | EF |
|---|---|---|---|
| F | 8 | 10 | 18 |
| I | 10 | 5 | 15 |

A atividade `J` somente poderá iniciar quando ambas terminarem. Logo:

```text
ES(J)=max(18,15)=18
```

Esse mecanismo é exatamente o cálculo do avanço direto do CPM.

A função `max_sum_time` implementa automaticamente essa propagação temporal dentro da rede.

---



<a id="sec-12"></a>

## 12. Execução da rede e algoritmo do CPM


### 12.1 Fundamentos de RPC: Regras executáveis e cálculo completo do CPM

A execução de uma RPC repete uma regra: observar a marcação, selecionar uma transição habilitada, consumir suas entradas e produzir suas saídas. As expressões de arco podem transportar os resultados de um cálculo para os próximos eventos.

Na integração com CPM, convém separar as operações:

| Operação | Cálculo necessário |
|---|---|
| Avanço direto | `EF = ES + duração`; `ES` da sucessora é o maior `EF` das predecessoras |
| Avanço reverso | `LS = LF - duração`; `LF` da predecessora é o menor `LS` das sucessoras |
| Folga total | `LS - ES`, equivalente a `LF - EF` |

Essas relações pressupõem o CPM usual, com durações definidas e sem restrições adicionais de recursos.

**O formalismo permite implementar esses cálculos, mas eles precisam estar nas expressões ou em um procedimento de análise.** Acumular tempos e tomar máximos corresponde ao avanço direto; isso, sozinho, não fornece os tempos mais tarde nem todas as folgas. Essa distinção ajuda a interpretar o alcance dos mecanismos descritos a seguir.


### 12.2 Algoritmo do CPM no Modelo

O modelo implementa implicitamente o algoritmo clássico do CPM através da propagação temporal dos tokens.

O avanço direto ocorre quando os tokens percorrem a rede acumulando:

```text
tempo + duração
```

A sincronização seleciona automaticamente o maior término acumulado entre fluxos concorrentes.

Assim, o modelo executa dinamicamente:

```text
EF = ES + duração
```

e:

```text
ES sucessora = max(EF predecessoras)
```

Em estruturas de convergência, o token sucessor somente é produzido quando o maior tempo acumulado é atingido.

O avanço reverso do CPM pode ser interpretado através da análise do tempo máximo admissível sem atrasar o projeto. Assim, o modelo permite analisar implicitamente:

```text
LF = LS + duração
```

e:

```text
Folga = LS - ES
```

As atividades pertencentes ao caminho crítico possuem folga igual a zero, ou seja, qualquer atraso nessas atividades impacta diretamente a duração total do projeto.

Portanto, a própria dinâmica da rede implementa o cálculo do caminho crítico.

---



<a id="sec-13"></a>

## 13. Execução de fluxos concorrentes


### 13.1 Fundamentos de RPC: Abertura de fluxos, concorrência e conflito

Uma transição pode consumir um token e produzir tokens em lugares diferentes, abrindo frentes de execução. Esses tokens podem conservar a mesma identificação de origem e carregar dados específicos de cada frente.

Compare três situações:

| Situação | Comportamento |
|---|---|
| Uma transição produz um token em cada um de dois lugares | Abre dois fluxos que podem evoluir independentemente |
| Duas transições disputam o único token de entrada | Há conflito: o disparo de uma pode desabilitar a outra |
| Uma transição exige tokens vindos de dois fluxos | Há sincronização: os dois precisam chegar |

**Concorrência** significa que execuções independentes são possíveis. O simulador pode apresentar seus disparos um a um; essa apresentação não transforma a lógica da rede em uma sequência obrigatória.

Também é preciso observar os recursos. Duas atividades logicamente independentes podem disputar a mesma equipe. A abertura de fluxos representa possibilidade de execução paralela; sua viabilidade operacional depende das demais condições da rede.


### 13.2 Paralelismo e Concorrência

Após a atividade `C`, o modelo permite múltiplos fluxos paralelos:

```text
C -> D
C -> E
C -> I
```

Isso representa concorrência operacional real.

Após a construção das paredes:

- telhado pode iniciar;
- instalações hidráulicas podem iniciar;
- instalações elétricas podem iniciar.

Esse paralelismo reduz o tempo total da obra e representa corretamente a dinâmica de execução em canteiros reais.

---



<a id="sec-14"></a>

## 14. Alocação e reutilização de recursos


### 14.1 Fundamentos de RPC: Reserva, ocupação e liberação de recursos

Um padrão de compartilhamento representa a disponibilidade por tokens. Para uma equipe indivisível, considere um lugar `Disponivel` com um token:

1. **Reservar:** a transição de início consome o token da atividade e o token da equipe disponível.
2. **Executar:** a atividade em andamento conserva a identificação da equipe ocupada.
3. **Liberar:** a transição de término produz a conclusão e devolve o recurso ao lugar `Disponivel`.

Se outra atividade precisa dessa equipe, ela aguarda enquanto o token de disponibilidade está ausente. Para recursos intercambiáveis, a multiplicidade pode representar a quantidade disponível.

Uma verificação útil é conferir se **recursos disponíveis + recursos ocupados** permanece igual à capacidade total, quando não há criação ou retirada de recursos no sistema.

Substituir a lista de equipes no valor de um token registra uma troca de associação. Para representar exclusividade e capacidade limitada, essa troca deve estar vinculada a regras de reserva e liberação como as descritas acima.


### 14.2 Compartilhamento de Recursos

O modelo também implementa troca e reutilização de equipes.

Exemplos identificados:

```text
F>J TROCA EQUIPE
E>H TROCA EQUIPE
```

Isso representa:

- realocação operacional;
- compartilhamento de recursos;
- restrições de capacidade;
- dependência entre frentes de trabalho.

Esse tipo de modelagem normalmente não aparece em exemplos acadêmicos básicos de Redes de Petri.

---



<a id="sec-15"></a>

## 15. Integração entre CPM e RPC


### 15.1 Fundamentos de RPC: Precedência, estado e análise de cenários

Ao relacionar um cronograma a uma RPC, é útil distinguir três perguntas:

| Pergunta | Informação necessária |
|---|---|
| O que precisa terminar antes? | Relações de precedência |
| O que pode ocorrer agora? | Marcação, valores dos tokens, guardas e recursos |
| Quando uma atividade pode começar? | Términos das predecessoras e demais restrições temporais ou operacionais |

O CPM também representa ramos paralelos e usa cálculos para analisar o prazo do projeto. A RPC acrescenta uma representação executável do estado, que pode incorporar disputas por recursos e outras regras.

Se houver capacidade limitada, o início efetivo pode superar o início mais cedo calculado apenas pelas precedências. Para comparar cenários, altere uma condição de cada vez e observe a duração resultante, as esperas e as atividades afetadas. Assim, a integração entre os métodos pode ser interpretada de acordo com as regras implementadas.


### 15.2 Relação entre CPM e CPN

O modelo implementa uma extensão dinâmica do CPM.

| CPM clássico | Modelo CPN |
|---|---|
| grafo estático | rede executável |
| cálculo matemático | simulação dinâmica |
| foco temporal | foco temporal e operacional |
| recursos implícitos | recursos explícitos |
| precedência | precedência + sincronização |
| sem concorrência explícita | concorrência explícita |

Assim, a CPN permite representar simultaneamente:

- lógica;
- tempo;
- recursos;
- concorrência;
- sincronização;
- propagação temporal.

---



<a id="sec-16"></a>

## 16. Integrações e aplicações avançadas


### 16.1 Fundamentos de RPC: Eventos, integração e verificação de propriedades

Uma RPC pode servir como referência formal para outros sistemas quando existe uma correspondência definida entre **objetos reais, tokens, eventos e transições**.

Por exemplo, uma informação de conclusão recebida do campo pode atualizar a condição representada na rede. Para isso, a integração precisa identificar a instância correta, interpretar o evento e verificar se ele é compatível com o estado atual.

A simulação permite explorar sequências de eventos. A análise do espaço de estados examina marcações alcançáveis e pode investigar propriedades como alcançar a conclusão ou encontrar bloqueios. Uma execução bem-sucedida mostra um comportamento possível; verificar uma propriedade exige analisar o conjunto de comportamentos pertinente.

Aplicações como monitoramento, BIM ou gêmeos digitais também exigem interfaces e dados externos. A rede fornece a lógica formal; a ligação com esses dados precisa ser implementada. [cpntools.org/category/documentation/doc-tasks-verification](https://cpntools.org/category/documentation/doc-tasks-verification/)


### 16.2 Aplicações Avançadas

Esse tipo de modelagem possui aplicações importantes em:

- BIM 4D;
- Process Mining;
- Digital Twins;
- simulação operacional;
- monitoramento de obras;
- planejamento inteligente;
- análise preditiva;
- controle de execução.

A integração entre CPN, CPM e Digital Twins permite transformar o cronograma em um modelo executável e monitorável em tempo real.

---



<a id="sec-17"></a>


## 17. Conclusão

O modelo apresentado demonstra como Redes de Petri Coloridas podem ser utilizadas para representar sistemas complexos de construção civil com elevado nível de realismo operacional.

O modelo implementa simultaneamente:

- precedência;
- propagação temporal;
- sincronização;
- compartilhamento de equipes;
- concorrência;
- restrições operacionais;
- mecanismos equivalentes ao CPM;
- cálculo dinâmico do caminho crítico.

Portanto, o modelo em rede Petri Colorida [GPCrev*.cpn](https://raw.githubusercontent.com/UniRobotica/Petri-nets/refs/heads/main/Exemplos/GPCrev9_Aula.cpn), disponível no repositório deste curso,  **aproxima-se significativamente da dinâmica real de execução de obras, permitindo representar não apenas o fluxo lógico das atividades, mas também os aspectos temporais e operacionais envolvidos no gerenciamento da construção civil**.



<a id="sec-18"></a>


## 18. Referências

- Jensen, K. *Colored Petri Nets*.
- Murata, T. *Petri Nets: Properties, Analysis and Applications*.
- van der Aalst, W. *Process Mining*.
- PMBOK 8ª edição.
- CPN Tools Documentation.

### 18.1 Fontes complementares dos fundamentos de RPC

Documentação técnica consultada para as explicações acrescentadas à aula. Acesso em: 9 nov. 2026.

- **F1 — Inscrições e expressões:** [visão geral](https://cpntools.org/2018/01/12/inscriptions-and-expressions/), [lugares](https://cpntools.org/2018/01/09/place-inscriptions/), [arcos](https://cpntools.org/2018/01/09/arc-inscriptions/) e [guardas](https://cpntools.org/2018/01/09/guards/).
- **F2 — Marcação e multiplicidade:** [Multisets](https://cpntools.org/2018/01/09/multisets/).
- **F3 — Vocabulário do modelo:** [Declarations](https://cpntools.org/2018/01/16/declarations/) e [Value declaration](https://cpntools.org/2018/01/09/value-declaration/).
- **F4 — Enumerações:** [Enumerated color sets](https://cpntools.org/2018/01/09/enumerated-color-sets/).
- **F5 — Tuplas e produtos:** [Product color sets](https://cpntools.org/2018/01/09/product-color-sets/).
- **F6 — Listas:** [List color sets](https://cpntools.org/2018/01/09/list-color-sets/).
- **F7 — Tipos temporizados:** [Timed color sets](https://cpntools.org/2018/01/09/timed-color-sets/).
- **F8 — Carimbos e atributos temporais:** [Timed multisets](https://cpntools.org/2018/01/09/timed-multisets/) e [Time attributes in tokens](https://cpntools.org/2018/01/09/time-attributes-in-tokens/).
- **F9 — Associação de valores:** [Variables](https://cpntools.org/2018/01/09/variables/).
- **F10 — Funções e padrões:** [Functions, declarations, and control structures](https://cpntools.org/2018/01/09/functions-declarations-and-control-structures/).
- **F11 — Verificação:** [State space analysis](https://cpntools.org/category/documentation/doc-tasks-verification/).
- **F12 — Padrões em SML:** Cornell University. [Lecture 3: Tuples. Records. Lists. Recursive Datatypes. Pattern Matching](https://www.cs.cornell.edu/courses/cs312/2004fa/lectures/lecture3.htm).


<a id="sec-19"></a>


## 19. APÊNDICE: Redes de Petri Coloridas e Linguagens de Programação

As **Redes de Petri Coloridas (RPC/CPN)** descrevem o comportamento de sistemas através da dinâmica entre **lugares**, **tokens** e **transições** — um paradigma fundamentalmente diferente da programação procedural ou orientada a objetos. O sistema não executa uma sequência fixa de comandos: ele *evolui* conforme transições vão sendo habilitadas pelo estado atual da rede.

Uma confusão frequente entre iniciantes é associar o CPN IDE ao HTML, porque os modelos são salvos em XML/PNML. Esse formato é apenas o mecanismo de serialização da estrutura gráfica da rede — a lógica do modelo é escrita em **Standard ML (SML)**, uma linguagem funcional estrita com tipagem estática e inferência de tipos. SML pertence à mesma família de OCaml, F# e Haskell, e não guarda relação com HTML.

### 19.1 Color Sets: tipagem dos tokens

Em SML/CPN, cada token carrega um valor de um tipo bem definido, chamado **color set**. Isso permite representar estados, leituras de sensores e comandos de forma semanticamente precisa — conceito análogo a `enum` e tipos algébricos em linguagens como Rust, OCaml e Ada.

In [ ]:
# Equivalente conceitual em Python — apenas para ilustração didática
# A sintaxe real é SML, executada no CPN IDE

# colset STATE = with Idle | Walk | Turn;   <- SML
# colset SENSOR = int;                      <- SML

from enum import Enum

class State(Enum):
    Idle = 0
    Walk = 1
    Turn = 2

# SENSOR é representado como int
sensor_value: int = 0

print("States:", [s.name for s in State])

### 19.2 Redes de Petri Coloridas e Linguagens de Programação

As **Redes de Petri Coloridas (RPC/CPN)** descrevem o comportamento de sistemas através da dinâmica entre **lugares**, **tokens** e **transições** — um paradigma fundamentalmente diferente da programação procedural ou orientada a objetos. O sistema não executa uma sequência fixa de comandos: ele *evolui* conforme transições vão sendo habilitadas pelo estado atual da rede.

Uma confusão frequente entre iniciantes é associar o CPN IDE ao HTML, porque os modelos são salvos em XML/PNML. Esse formato é apenas o mecanismo de serialização da estrutura gráfica da rede — a lógica do modelo é escrita em **Standard ML (SML)**, uma linguagem funcional estrita com tipagem estática e inferência de tipos. SML pertence à mesma família de OCaml, F# e Haskell, e não guarda relação com HTML.

### 19.3 Color Sets: tipagem dos tokens

Em SML/CPN, cada token carrega um valor de um tipo bem definido, chamado **color set**. Isso permite representar estados, leituras de sensores e comandos de forma semanticamente precisa — conceito análogo a `enum` e tipos algébricos em linguagens como Rust, OCaml e Ada.

In [ ]:
# Equivalente conceitual em Python — apenas para ilustração didática
# A sintaxe real é SML, executada no CPN IDE

# colset STATE = with Idle | Walk | Turn;   <- SML
# colset SENSOR = int;                      <- SML

from enum import Enum

class State(Enum):
    Idle = 0
    Walk = 1
    Turn = 2

# SENSOR é representado como int
sensor_value: int = 0

print("States:", [s.name for s in State])

### 19.4 Guardas e funções de arco

As transições nas RPC podem ter **guardas** — condições booleanas que determinam se a transição está habilitada. Em SML, essas funções seguem um estilo declarativo: definem *o que* é verdadeiro, não *como* calcular passo a passo.

A comparação abaixo mostra a mesma lógica de detecção de obstáculo nas três linguagens. Note que em SML o corpo da função *é* a expressão — não há `return` explícito.

In [ ]:
# Python
def detect_obstacle(x: int) -> bool:
    return x < 10

# Equivalente em SML (CPN IDE):
# fun detectObstacle(x:int) = x < 10;

# Equivalente em C:
# bool detectObstacle(int x) { return x < 10; }

# Teste
for dist in [5, 10, 15]:
    print(f"distância={dist}cm → obstáculo: {detect_obstacle(dist)}")

A sintaxe é superficialmente próxima, mas o papel dessas funções nas RPC é diferente: elas não são chamadas explicitamente pelo programador, mas avaliadas *pelo motor de simulação* a cada tentativa de disparo de transição.

### 19.5 Concorrência

Uma das vantagens centrais das RPC é a modelagem natural de **comportamento concorrente**. No [robô Otto](https://www.ottodiy.com/), por exemplo, a caminhada e a leitura do sensor ultrassônico ocorrem simultaneamente — situação difícil de representar com clareza em código procedural, mas direta em uma rede de Petri: dois fluxos de tokens coexistem na rede e interagem apenas quando uma transição compartilhada é habilitada.

Quando o sensor registra distância abaixo do limiar, um token com valor crítico alcança o lugar de entrada de uma transição `Walk → Turn`, que se habilita e redireciona o estado do robô. Não há `if` global nem loop de controle central — a sincronização emerge da topologia da própria rede.

In [ ]:
# Simulação simplificada da dinâmica de tokens — apenas ilustração
# A rede real seria definida graficamente no CPN IDE

import random

state = "Walk"
steps = 0

print(f"{'Passo':<8} {'Sensor (cm)':<14} {'Estado'}")
print("-" * 35)

while steps < 8:
    sensor = random.randint(5, 25)
    if state == "Walk" and detect_obstacle(sensor):
        state = "Turn"      # transição Walk→Turn habilitada
    elif state == "Turn":
        state = "Walk"      # transição Turn→Walk habilitada após girar
    print(f"{steps:<8} {sensor:<14} {state}")
    steps += 1

### 19.6 Arquitetura: RPC + microcontrolador

Uma prática consolidada em robótica embarcada é usar as RPC como **camada de especificação formal** e o microcontrolador como **executor físico**. A rede descreve e valida o comportamento — estados, transições, condições de disparo — antes mesmo de qualquer linha de firmware ser escrita. O microcontrolador recebe então uma especificação verificada formalmente e implementa os comandos concretos: controle dos servos, leitura do ultrassônico e comunicação.

Essa separação reduz erros de lógica concorrente e facilita a verificação de propriedades como ausência de deadlock e alcançabilidade de estados — análises que o CPN IDE realiza automaticamente sobre o modelo.